<a href="https://colab.research.google.com/github/cardtnn/ACO_VRP/blob/main/Copy_of_Ant_Colony_Algorithm_V0_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
import random
import time


# =========================================================
# BASIC FUNCTIONS
# =========================================================

def euclidean(a, b):
    """Euclidean distance between two coordinates."""
    return math.hypot(a[0] - b[0], a[1] - b[1])


def build_distance_matrix(coords):
    """Create distance matrix."""
    n = len(coords)

    return [
        [
            euclidean(coords[i], coords[j])
            for j in range(n)
        ]
        for i in range(n)
    ]


def route_distance(route, dist):
    """Calculate distance of one route."""
    total = 0

    for i in range(len(route) - 1):
        total += dist[route[i]][route[i + 1]]

    return total


def solution_distance(routes, dist):
    """Calculate total distance of all teams."""
    return sum(route_distance(route, dist) for route in routes)


# =========================================================
# ACO CHOICE
# =========================================================

def roulette_choice(candidates, weights, rng):

    total = sum(weights)

    r = rng.random() * total

    cumulative = 0

    for candidate, weight in zip(candidates, weights):

        cumulative += weight

        if r <= cumulative:
            return candidate

    return candidates[-1]


def choose_next(
    current,
    unvisited,
    pheromone,
    dist,
    alpha,
    beta,
    rng
):

    candidates = list(unvisited)

    weights = []

    for j in candidates:

        # pheromone information
        tau = max(
            pheromone[current][j],
            1e-12
        )

        # heuristic information = inverse distance
        eta = 1.0 / max(
            dist[current][j],
            1e-12
        )

        weight = (
            (tau ** alpha)
            *
            (eta ** beta)
        )

        weights.append(weight)

    return roulette_choice(
        candidates,
        weights,
        rng
    )


# =========================================================
# PHEROMONE UPDATE
# =========================================================

def evaporate(pheromone, rho):

    n = len(pheromone)

    for i in range(n):
        for j in range(n):

            pheromone[i][j] *= (1 - rho)

            pheromone[i][j] = max(
                pheromone[i][j],
                1e-12
            )


def deposit_iteration_best(
    pheromone,
    routes,
    total_distance,
    Q=1.0
):

    # shorter solution -> larger pheromone deposit
    delta = Q / total_distance

    for route in routes:

        for i in range(len(route) - 1):

            a = route[i]
            b = route[i + 1]

            pheromone[a][b] += delta
            pheromone[b][a] += delta


# =========================================================
# APPROACH A
# DIRECT MULTI-ROUTE ACO
# =========================================================

def aco_direct_multiroute(
    coords,
    n_teams,
    capacity,
    ants=20,
    iterations=300,
    alpha=1,
    beta=2,
    rho=0.1,
    seed=0
):

    rng = random.Random(seed)

    dist = build_distance_matrix(coords)

    n = len(coords)

    # Node 0 = Depot
    # Nodes 1...n-1 = Jobs
    jobs = set(range(1, n))

    # Maximum number of jobs that can be served today
    max_served = min(
        len(jobs),
        n_teams * capacity
    )

    pheromone = [
        [1.0 for _ in range(n)]
        for _ in range(n)
    ]

    global_best_routes = None
    global_best_distance = float("inf")
    global_best_unserved = None

    for iteration in range(iterations):

        iteration_best_routes = None
        iteration_best_distance = float("inf")

        for ant in range(ants):

            unvisited = set(jobs)

            routes = []

            served_count = 0

            # Build route for each team
            for team in range(n_teams):

                route = [0]
                current = 0

                for _ in range(capacity):

                    # Stop if all available capacity is already used
                    if (
                        not unvisited
                        or served_count >= max_served
                    ):
                        break

                    next_job = choose_next(
                        current,
                        unvisited,
                        pheromone,
                        dist,
                        alpha,
                        beta,
                        rng
                    )

                    route.append(next_job)

                    unvisited.remove(next_job)

                    current = next_job

                    served_count += 1

                route.append(0)

                routes.append(route)

            total_distance = solution_distance(
                routes,
                dist
            )

            # Iteration best
            if total_distance < iteration_best_distance:

                iteration_best_distance = total_distance

                iteration_best_routes = [
                    r[:] for r in routes
                ]

            # Global best
            if total_distance < global_best_distance:

                global_best_distance = total_distance

                global_best_routes = [
                    r[:] for r in routes
                ]

                global_best_unserved = sorted(
                    unvisited
                )

        # Pheromone evaporation
        evaporate(
            pheromone,
            rho
        )

        # Iteration-best deposits pheromone
        deposit_iteration_best(
            pheromone,
            iteration_best_routes,
            iteration_best_distance
        )

    return (
        global_best_routes,
        global_best_distance,
        global_best_unserved
    )

# =========================================================
# APPROACH B
# GIANT TOUR + FIXED SPLIT
# =========================================================

def fixed_split(
    ordering,
    n_teams,
    capacity
):

    routes = []

    index = 0

    for team in range(n_teams):

        chunk = ordering[
            index:index + capacity
        ]

        index += capacity

        route = (
            [0]
            +
            chunk
            +
            [0]
        )

        routes.append(route)

    return routes


def aco_giant_tour_split(
    coords,
    n_teams,
    capacity,
    ants=20,
    iterations=300,
    alpha=1,
    beta=2,
    rho=0.1,
    seed=0
):

    rng = random.Random(seed)

    dist = build_distance_matrix(coords)

    n = len(coords)

    jobs = set(range(1, n))

    max_served = min(
        len(jobs),
        n_teams * capacity
    )

    pheromone = [
        [1.0 for _ in range(n)]
        for _ in range(n)
    ]

    global_best_order = None
    global_best_routes = None
    global_best_distance = float("inf")
    global_best_unserved = None

    for iteration in range(iterations):

        iteration_best_routes = None
        iteration_best_distance = float("inf")

        for ant in range(ants):

            unvisited = set(jobs)

            ordering = []

            current = 0

            # Build only as many jobs as can actually be served
            while (
                unvisited
                and len(ordering) < max_served
            ):

                next_job = choose_next(
                    current,
                    unvisited,
                    pheromone,
                    dist,
                    alpha,
                    beta,
                    rng
                )

                ordering.append(next_job)

                unvisited.remove(next_job)

                current = next_job

            routes = fixed_split(
                ordering,
                n_teams,
                capacity
            )

            total_distance = solution_distance(
                routes,
                dist
            )

            if total_distance < iteration_best_distance:

                iteration_best_distance = total_distance

                iteration_best_routes = [
                    r[:] for r in routes
                ]

            if total_distance < global_best_distance:

                global_best_distance = total_distance

                global_best_order = ordering[:]

                global_best_routes = [
                    r[:] for r in routes
                ]

                global_best_unserved = sorted(
                    unvisited
                )

        evaporate(
            pheromone,
            rho
        )

        deposit_iteration_best(
            pheromone,
            iteration_best_routes,
            iteration_best_distance
        )

    return (
        global_best_order,
        global_best_routes,
        global_best_distance,
        global_best_unserved
    )

# =========================================================
# TOY DATASET
# =========================================================

# Node 0 = Depot
# Node 1-8 = Jobs

coords = [

    (0, 0),      # Depot

    (2, 8),      # J1
    (4, 7),      # J2
    (8, 8),      # J3
    (9, 3),      # J4
    (7, 1),      # J5

    (3, 2),      # J6
    (1, 4),      # J7
    (6, 5),      # J8
    (11, 7),     # J9
    (12, 2),     # J10

    (5, 10),     # J11
    (10, 10),    # J12
    (14, 5),     # J13
    (2, 12),     # J14
    (6, 12),     # J15

    (13, 9),     # J16
    (15, 2),     # J17
    (9, 12),     # J18
    (3, 14),     # J19
    (16, 8)      # J20
]


N_TEAMS = 8
CAPACITY = 2

ANTS = 20
ITERATIONS = 300

ALPHA = 1
BETA = 2
RHO = 0.1

SEED = 1


# =========================================================
# RUN APPROACH A
# =========================================================

start = time.perf_counter()

routes_A, distance_A, unserved_A = aco_direct_multiroute(
    coords,
    n_teams=N_TEAMS,
    capacity=CAPACITY,
    ants=ANTS,
    iterations=ITERATIONS,
    alpha=ALPHA,
    beta=BETA,
    rho=RHO,
    seed=SEED
)

runtime_A = time.perf_counter() - start


print("\n==============================")
print("APPROACH A: DIRECT MULTI-ROUTE")
print("==============================")

for team, route in enumerate(routes_A, start=1):

    print(
        f"Team {team}: "
        +
        " -> ".join(
            "Depot" if node == 0 else f"J{node}"
            for node in route
        )
    )

print(f"\nUnserved Jobs: {unserved_A}")
print(f"Total distance: {distance_A:.2f}")
print(f"Runtime: {runtime_A:.4f} seconds")

# =========================================================
# RUN APPROACH B
# =========================================================

start = time.perf_counter()

order_B, routes_B, distance_B, unserved_B = (
    aco_giant_tour_split(
        coords,
        n_teams=N_TEAMS,
        capacity=CAPACITY,
        ants=ANTS,
        iterations=ITERATIONS,
        alpha=ALPHA,
        beta=BETA,
        rho=RHO,
        seed=SEED
    )
)

runtime_B = time.perf_counter() - start


print("\n==============================")
print("APPROACH B: GIANT TOUR + SPLIT")
print("==============================")

print(
    "\nServed ordering:",
    " -> ".join(
        f"J{x}" for x in order_B
    )
)

print()

for team, route in enumerate(routes_B, start=1):

    print(
        f"Team {team}: "
        +
        " -> ".join(
            "Depot" if node == 0 else f"J{node}"
            for node in route
        )
    )

print(f"\nUnserved Jobs: {unserved_B}")
print(f"Total distance: {distance_B:.2f}")
print(f"Runtime: {runtime_B:.4f} seconds")


APPROACH A: DIRECT MULTI-ROUTE
Team 1: Depot -> J8 -> J5 -> Depot
Team 2: Depot -> J18 -> J12 -> Depot
Team 3: Depot -> J7 -> J6 -> Depot
Team 4: Depot -> J2 -> J1 -> Depot
Team 5: Depot -> J15 -> J11 -> Depot
Team 6: Depot -> J3 -> J9 -> Depot
Team 7: Depot -> J4 -> J10 -> Depot
Team 8: Depot -> J14 -> J19 -> Depot

Unserved Jobs: [13, 16, 17, 20]
Total distance: 187.37
Runtime: 0.8430 seconds

APPROACH B: GIANT TOUR + SPLIT

Served ordering: J6 -> J7 -> J2 -> J1 -> J14 -> J19 -> J11 -> J15 -> J18 -> J12 -> J3 -> J8 -> J4 -> J5 -> J10 -> J17

Team 1: Depot -> J6 -> J7 -> Depot
Team 2: Depot -> J2 -> J1 -> Depot
Team 3: Depot -> J14 -> J19 -> Depot
Team 4: Depot -> J11 -> J15 -> Depot
Team 5: Depot -> J18 -> J12 -> Depot
Team 6: Depot -> J3 -> J8 -> Depot
Team 7: Depot -> J4 -> J5 -> Depot
Team 8: Depot -> J10 -> J17 -> Depot

Unserved Jobs: [9, 13, 16, 20]
Total distance: 188.45
Runtime: 1.6740 seconds


In [ ]:
import statistics
from collections import Counter

N_RUNS = 30

distances_A = []
distances_B = []

runtimes_A = []
runtimes_B = []

unserved_counter_A = Counter()
unserved_counter_B = Counter()


for seed in range(N_RUNS):

    # =================================
    # Approach A
    # =================================

    start = time.perf_counter()

    routes_A, distance_A, unserved_A = (
        aco_direct_multiroute(
            coords,
            n_teams=N_TEAMS,
            capacity=CAPACITY,
            ants=ANTS,
            iterations=ITERATIONS,
            alpha=ALPHA,
            beta=BETA,
            rho=RHO,
            seed=seed
        )
    )

    runtime_A = time.perf_counter() - start

    distances_A.append(distance_A)
    runtimes_A.append(runtime_A)

    for job in unserved_A:
        unserved_counter_A[job] += 1


    # =================================
    # Approach B
    # =================================

    start = time.perf_counter()

    order_B, routes_B, distance_B, unserved_B = (
        aco_giant_tour_split(
            coords,
            n_teams=N_TEAMS,
            capacity=CAPACITY,
            ants=ANTS,
            iterations=ITERATIONS,
            alpha=ALPHA,
            beta=BETA,
            rho=RHO,
            seed=seed
        )
    )

    runtime_B = time.perf_counter() - start

    distances_B.append(distance_B)
    runtimes_B.append(runtime_B)

    for job in unserved_B:
        unserved_counter_B[job] += 1


# =================================
# SUMMARY
# =================================

print("\n==============================")
print("30-RUN SUMMARY")
print("==============================")


print("\nApproach A")

print(
    f"Best distance : "
    f"{min(distances_A):.4f}"
)

print(
    f"Worst distance: "
    f"{max(distances_A):.4f}"
)

print(
    f"Mean distance : "
    f"{statistics.mean(distances_A):.4f}"
)

print(
    f"Std distance  : "
    f"{statistics.stdev(distances_A):.4f}"
)

print(
    f"Mean runtime  : "
    f"{statistics.mean(runtimes_A):.6f} sec"
)


print("\nApproach B")

print(
    f"Best distance : "
    f"{min(distances_B):.4f}"
)

print(
    f"Worst distance: "
    f"{max(distances_B):.4f}"
)

print(
    f"Mean distance : "
    f"{statistics.mean(distances_B):.4f}"
)

print(
    f"Std distance  : "
    f"{statistics.stdev(distances_B):.4f}"
)

print(
    f"Mean runtime  : "
    f"{statistics.mean(runtimes_B):.6f} sec"
)


# =================================
# UNSERVED FREQUENCY
# =================================

print("\n==============================")
print("UNSERVED FREQUENCY")
print("==============================")


print("\nApproach A")

for job in range(1, len(coords)):

    print(
        f"J{job}: "
        f"{unserved_counter_A[job]} / {N_RUNS}"
    )


print("\nApproach B")

for job in range(1, len(coords)):

    print(
        f"J{job}: "
        f"{unserved_counter_B[job]} / {N_RUNS}"
    )


30-RUN SUMMARY

Approach A
Best distance : 187.3655
Worst distance: 187.4108
Mean distance : 187.3912
Std distance  : 0.0228
Mean runtime  : 0.733316 sec

Approach B
Best distance : 187.3655
Worst distance: 188.4462
Mean distance : 187.9442
Std distance  : 0.5445
Mean runtime  : 0.697774 sec

UNSERVED FREQUENCY

Approach A
J1: 0 / 30
J2: 0 / 30
J3: 0 / 30
J4: 0 / 30
J5: 0 / 30
J6: 0 / 30
J7: 0 / 30
J8: 0 / 30
J9: 0 / 30
J10: 0 / 30
J11: 0 / 30
J12: 0 / 30
J13: 30 / 30
J14: 0 / 30
J15: 0 / 30
J16: 30 / 30
J17: 13 / 30
J18: 17 / 30
J19: 0 / 30
J20: 30 / 30

Approach B
J1: 0 / 30
J2: 0 / 30
J3: 0 / 30
J4: 0 / 30
J5: 0 / 30
J6: 0 / 30
J7: 0 / 30
J8: 0 / 30
J9: 14 / 30
J10: 0 / 30
J11: 0 / 30
J12: 0 / 30
J13: 30 / 30
J14: 0 / 30
J15: 0 / 30
J16: 28 / 30
J17: 14 / 30
J18: 4 / 30
J19: 0 / 30
J20: 30 / 30
